# Multichromatic illumination: fundamental and second harmonic

A synchrotron exposure can contain mutually incoherent colors. Here 95% of the
incident photon flux is at 780 eV and 5% at 1560 eV. Each color uses its own optical
constants, multislice wavelength and detector projection. Their detector
**intensities** add; complex fields do not interfere. Sensor noise and artifacts
are applied once to the combined exposure.

Use the same experiment declaration and simulate/save/load workflow as Tutorial 00.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/scattering_calculator').exists())
sys.path.insert(0, str(ROOT / 'src'))
from scattering_calculator import simulation_pipelines as sim
OUT = ROOT / 'outputs' / '19_multichromatic_illumination'


In [ ]:
# 1. X-RAY SOURCE — nominal energy in eV, total flux in photons/s
# 2. SIMULATION GRID — (y, x) pixels, pitch in metres
# 3. SAMPLE — recipe thickness literals in nm
# 4. MAGNETIZATION — pattern lengths in metres, angles in radians
# 5. APERTURES — no drilled openings for this thin-film example
# 6. ILLUMINATION — simultaneous spectral components, not scan alternatives
# 7. SAMPLE PROPAGATION — independently applied to every color
# 8. DETECTOR / ACQUISITION — one exposure of the summed intensity
# 9. BEAMSTOP; 10. OUTPUTS
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=780., photon_flux=1e8, pol='LH'),
    simulation=sim.SimulationConfig(shape=(128,128), real_space_pixel_size=4e-9),
    sample=sim.SampleConfig(recipe='Co(30)', max_slice_thickness=5e-9),
    magnetic_pattern=sim.MagneticPatternConfig(pattern_type_method='magnon_wave',
        pattern_config={'period':40e-9, 'amplitude':.1, 'angle':np.deg2rad(30.)}),
    aperture=sim.FrontApertureConfig(aperture_method=None),
    illumination=sim.IlluminationConfig(illumination_function='gaussian',
        illumination_config={'center':(0.,0.), 'distance':0., 'fwhm':150e-9, 'alpha_beam':(0.,0.)},
        spectral_components=(
            sim.SpectralComponentConfig(energy_factor=1., weight=.95),
            sim.SpectralComponentConfig(energy_factor=2., weight=.05),
        )),
    propagation=sim.SamplePropagatorConfig(propagator_method='Jones',
        propagator_config={'propagate':True, 'farfield_oversampling':2}),
    detector=sim.DetectorConfig(shape=(128,128), pixel_size=13.5e-6,
        sample_to_detector_distance=.03, detector_center=(64,64),
        projection_energy=None,
        measurement_config={'exposure_time':.4, 'number_frames':1, 'max_counts_per_image':None},
        detector_params={'noise_seed':17, 'counts_per_photon':1., 'quantum_efficiency':.8,
                         'readout_noise_average':0., 'readout_noise_sigma':2., 'detector_threshold':64e3},
        artifacts_config={'sigma_photon':0.}),
    beamstop=sim.BeamstopConfig(bs_method=None),
    outputs=sim.OutputConfig(path=OUT/'experiment.h5', overwrite=True,
        save=('exit_wave','fft_intensity','detector_ideal','detector_measured')),
)


Weights are relative **photon-flux** weights, normalized to sum to one;
`xray.photon_flux` is the total. Absolute energies can instead be specified with
`SpectralComponentConfig(energy=1560., weight=.05)`. Energy factors follow each
nominal scan energy, allowing a scan of a fundamental plus its harmonics.

Keep `detector.projection_energy=None` for the physical detector projection of
each color. A fixed projection energy remains available as a q-coordinate
comparison convention.


In [ ]:
results = sim.simulate_experiment(experiment)


In [ ]:
results = sim.load_results(experiment.outputs.path)
summary = sim.plot_results(results, save_path=OUT/'summary.png')


In [ ]:
fundamental = results.read('spectral_components/000000/detector_ideal')
harmonic = results.read('spectral_components/000001/detector_ideal')
combined = results.read('detector_ideal')
np.testing.assert_allclose(combined, fundamental + harmonic, rtol=1e-12)

x = results.read('coordinates/detector_x') * 1e3
y = results.read('coordinates/detector_y') * 1e3
extent = [x.min(), x.max(), y.min(), y.max()]
vmax = combined.max()
floor = vmax * 1e-7
fig, axes = plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
for ax, image, title in zip(axes, (fundamental,harmonic,combined),
        ('780 eV — 95% incident photons','1560 eV — 5% incident photons','Combined ideal detector')):
    im = ax.imshow(np.maximum(image,floor),origin='lower',extent=extent,
                   norm=LogNorm(vmin=floor,vmax=vmax))
    ax.set(title=title,xlabel='Detector x (mm)',ylabel='Detector y (mm)')
fig.colorbar(im, ax=axes, label='Incident photons/s per detector pixel')
fig.savefig(OUT/'harmonic_comparison.png',dpi=160)


The color axis is retained in saved complex fields:
`exit_wave[..., 0]` is the flux-weighted fundamental field and
`exit_wave[..., 1]` the flux-weighted harmonic field. For Jones propagation,
shape is `(y, x, polarization, color)`. Sum squared amplitudes across polarization
and color for exit-plane intensity; never sum the complex color amplitudes.

Each `spectral_components/NNNNNN` group saves the actual energy, polarization,
photon-flux fraction, optical constants and q coordinates. Those q coordinates
differ for the two colors even though the physical camera pixels are shared.
